In [ ]:
!pip install segmentation-models-pytorch==0.3.3 timm openpyxl --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.7/106.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 26.2 MB/s eta 0:00:00


In [ ]:
import sys
import os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive"):
        drive.mount("/content/drive")
# ==============================================================================
# Ô CODE 1: IMPORT VÀ SETUP
# ==============================================================================
print("="*80)
print("🔬 BƯỚC 7: ĐÁNH GIÁ TOÀN BỘ PIPELINE & BÁO CÁO (WITH VERSIONING)")
print("="*80)

import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, json, cv2
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import segmentation_models_pytorch as smp
import timm

from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_recall_fscore_support, roc_auc_score, roc_curve
)
from sklearn.preprocessing import label_binarize
import seaborn as sns
from datetime import datetime


GDRIVE_PATH = "d:/DoAn_DaLieu"
CHECKPOINT_PATH = os.path.join(GDRIVE_PATH, "3_Checkpoints")
RESULTS_PATH = os.path.join(GDRIVE_PATH, "5_Results")
RESULTS_ARCHIVE_PATH = os.path.join(RESULTS_PATH, "archive")  # ← THÊM MỚI
os.makedirs(RESULTS_PATH, exist_ok=True)
os.makedirs(RESULTS_ARCHIVE_PATH, exist_ok=True)  # ← THÊM MỚI

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🖥️ Device: {device}")

🔬 BƯỚC 7: ĐÁNH GIÁ TOÀN BỘ PIPELINE & BÁO CÁO (WITH VERSIONING)


ValueError: mount failed

In [ ]:
# ==============================================================================
# Ô CODE 2: VERSIONING HELPER FUNCTIONS
# ==============================================================================
print("\n" + "="*80)
print("🔄 VERSIONING SYSTEM")
print("="*80)

def get_version_timestamp():
    """Generate version timestamp"""
    return datetime.now().strftime('%Y%m%d_%H%M%S')

def archive_old_report(report_path, archive_path):
    """Archive existing report before creating new one"""
    if os.path.exists(report_path):
        import shutil
        filename = os.path.basename(report_path)
        name, ext = os.path.splitext(filename)

        # Find existing archived versions
        existing_versions = [f for f in os.listdir(archive_path)
                           if f.startswith(name) and f.endswith(ext)]
        version_num = len(existing_versions) + 1

        # Archive with version number
        archived_name = f"{name}_v{version_num:03d}_{get_version_timestamp()}{ext}"
        archived_path = os.path.join(archive_path, archived_name)

        shutil.copy2(report_path, archived_path)
        print(f"   📦 Archived: {archived_name}")
        return archived_path
    return None

def save_with_versioning(content, filename, archive=True):
    """
    Save file with versioning support
    - Always update 'latest' version
    - Optionally archive old version
    """
    filepath = os.path.join(RESULTS_PATH, filename)

    # Archive old version if exists
    if archive and os.path.exists(filepath):
        archive_old_report(filepath, RESULTS_ARCHIVE_PATH)

    # Save new version
    if filename.endswith('.txt'):
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(content)
    elif filename.endswith('.xlsx'):
        # For Excel, content is the writer object
        pass  # Handled separately
    elif filename.endswith('.json'):
        with open(filepath, 'w') as f:
            json.dump(content, f, indent=4)

    print(f"   ✅ Updated: {filename}")

    return filepath

print("✅ Versioning functions ready")

In [ ]:
# ==============================================================================
# Ô CODE 3: KIỂM TRA CÁC BƯỚC ĐÃ HOÀN THÀNH
# ==============================================================================
print("\n" + "="*80)
print("📋 KIỂM TRA CÁC BƯỚC ĐÃ HOÀN THÀNH")
print("="*80)

required_checkpoints = {
    "01": "01_prepare_and_split_complete.json",
    "02": "02_unet_complete.json",
    "03": "03_deeplabv3plus_complete.json",
    "04": "04_hybrid_complete.json",
    "05": "05_roi_extraction_complete.json",
    "06": "06_classification_complete.json"
}

all_checkpoints = {}
missing_steps = []

for step, filename in required_checkpoints.items():
    filepath = os.path.join(CHECKPOINT_PATH, filename)
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            all_checkpoints[step] = json.load(f)
        print(f"✅ Bước {step}: {filename}")
    else:
        print(f"❌ Bước {step}: THIẾU {filename}")
        missing_steps.append(step)

if missing_steps:
    raise FileNotFoundError(
        f"❌ Thiếu các bước: {', '.join(missing_steps)}\n"
        f"Vui lòng chạy các file tương ứng trước!"
    )

print("\n✅ Tất cả các bước đã hoàn thành!")

In [ ]:
# ==============================================================================
# Ô CODE 4: THU THẬP KẾT QUẢ
# ==============================================================================
print("\n" + "="*80)
print("📊 THU THẬP KẾT QUẢ TỪ CÁC BƯỚC")
print("="*80)

dataset_info = all_checkpoints["01"]["datasets"]
unet_res = all_checkpoints["02"]["results"]
deeplab_res = all_checkpoints["03"]["results"]
hybrid_results = all_checkpoints["04"]["results"]
best_seg_model = all_checkpoints["04"]["best_model"]
roi_info = all_checkpoints["05"]["roi_data"]
cls_results = all_checkpoints["06"]["results"]
cls_config = all_checkpoints["06"]["config"]

seg_info = dataset_info['segmentation']
cls_info = dataset_info['classification']

best_dice = max([m['dice'] for m in hybrid_results.values()])
best_iou = max([m['iou'] for m in hybrid_results.values()])

print(f"✅ Đã thu thập tất cả kết quả")
print(f"   Best Segmentation: {best_seg_model}")
print(f"   Best Dice: {best_dice:.4f}")
print(f"   Classification Acc: {cls_results['test_acc']:.2f}%")

In [ ]:
# ==============================================================================
# Ô CODE 5: CREATE VISUALIZATIONS (Same as before)
# ==============================================================================
print("\n" + "="*80)
print("📊 TẠO VISUALIZATIONS")
print("="*80)

# Segmentation Comparison
seg_comparison_path = os.path.join(RESULTS_PATH, "segmentation_comparison.png")
archive_old_report(seg_comparison_path, RESULTS_ARCHIVE_PATH)

models = ['U-Net', 'DeepLabV3+'] + list(hybrid_results.keys())
dice_scores = [unet_res['test_dice'], deeplab_res['test_dice']] + [m['dice'] for m in hybrid_results.values()]
iou_scores = [unet_res['test_iou'], deeplab_res['test_iou']] + [m['iou'] for m in hybrid_results.values()]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(len(models))
colors = ['steelblue', 'coral', 'mediumseagreen', 'gold'][:len(models)]

# Dice
axes[0].bar(x, dice_scores, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[0].set_xlabel('Model', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Dice Score', fontsize=12, fontweight='bold')
axes[0].set_title('Dice Score Comparison', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=15, ha='right')
axes[0].grid(axis='y', alpha=0.3, linestyle='--')
for i, v in enumerate(dice_scores):
    axes[0].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=10, fontweight='bold')

# IoU
axes[1].bar(x, iou_scores, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[1].set_xlabel('Model', fontsize=12, fontweight='bold')
axes[1].set_ylabel('IoU Score', fontsize=12, fontweight='bold')
axes[1].set_title('IoU Score Comparison', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, rotation=15, ha='right')
axes[1].grid(axis='y', alpha=0.3, linestyle='--')
for i, v in enumerate(iou_scores):
    axes[1].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=10, fontweight='bold')

best_idx = models.index(best_seg_model)
axes[0].patches[best_idx].set_edgecolor('red')
axes[0].patches[best_idx].set_linewidth(3)
axes[1].patches[best_idx].set_edgecolor('red')
axes[1].patches[best_idx].set_linewidth(3)

plt.suptitle('Segmentation Models Performance Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(seg_comparison_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"✅ Saved: segmentation_comparison.png")

# Pipeline Performance
pipeline_summary_path = os.path.join(RESULTS_PATH, "pipeline_performance.png")
archive_old_report(pipeline_summary_path, RESULTS_ARCHIVE_PATH)

fig = plt.figure(figsize=(18, 6))
gs = fig.add_gridspec(1, 3, hspace=0.3, wspace=0.3)

# Panel 1: Segmentation
ax1 = fig.add_subplot(gs[0, 0])
seg_metrics = ['Dice', 'IoU']
seg_values = [best_dice, best_iou]
bars1 = ax1.bar(seg_metrics, seg_values, color=['steelblue', 'coral'], alpha=0.8, edgecolor='black', linewidth=2, width=0.6)
ax1.set_ylim([0.7, 1.0])
ax1.set_ylabel('Score', fontsize=12, fontweight='bold')
ax1.set_title(f'Best Segmentation\n({best_seg_model})', fontsize=13, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
for bar, val in zip(bars1, seg_values):
    ax1.text(bar.get_x() + bar.get_width()/2., val + 0.01, f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

# Panel 2: Classification
ax2 = fig.add_subplot(gs[0, 1])
cls_metrics = ['Test Acc', 'Best Val Acc']
cls_values = [cls_results['test_acc'], cls_results['best_val_acc']]
bars2 = ax2.bar(cls_metrics, cls_values, color=['mediumseagreen', 'gold'], alpha=0.8, edgecolor='black', linewidth=2, width=0.6)
ax2.set_ylim([0, 100])
ax2.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax2.set_title('Classification Performance\n(EfficientNet + Attention)', fontsize=13, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
for bar, val in zip(bars2, cls_values):
    ax2.text(bar.get_x() + bar.get_width()/2., val + 2, f'{val:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)

# Panel 3: ROI Statistics
ax3 = fig.add_subplot(gs[0, 2])
roi_splits = ['Train', 'Val', 'Test']
roi_counts = [roi_info['train'], roi_info['val'], roi_info['test']]
bars3 = ax3.bar(roi_splits, roi_counts, color=['mediumpurple', 'lightcoral', 'lightskyblue'], alpha=0.8, edgecolor='black', linewidth=2, width=0.6)
ax3.set_ylabel('Number of ROIs', fontsize=12, fontweight='bold')
ax3.set_title('ROI Extraction Statistics', fontsize=13, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)
for bar, val in zip(bars3, roi_counts):
    ax3.text(bar.get_x() + bar.get_width()/2., val + max(roi_counts)*0.02, f'{val}', ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.suptitle('COMPLETE PIPELINE PERFORMANCE SUMMARY', fontsize=16, fontweight='bold', y=1.02)
plt.savefig(pipeline_summary_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"✅ Saved: pipeline_performance.png")

In [ ]:
# ==============================================================================
# Ô CODE 6: CREATE EXCEL SUMMARY WITH VERSIONING
# ==============================================================================
print("\n" + "="*80)
print("📊 TẠO EXCEL SUMMARY (WITH VERSIONING)")
print("="*80)

excel_filename = "FINAL_REPORT.xlsx"
excel_path = os.path.join(RESULTS_PATH, excel_filename)

# Archive old Excel if exists
archive_old_report(excel_path, RESULTS_ARCHIVE_PATH)

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    # Sheet 1: Dataset Info
    dataset_summary = pd.DataFrame({
        'Dataset': ['Segmentation', 'Classification'],
        'Total': [seg_info['total'], cls_info['total']],
        'Train': [seg_info['train'], cls_info['train']],
        'Val': [seg_info['val'], cls_info['val']],
        'Test': [seg_info['test'], cls_info['test']]
    })
    dataset_summary.to_excel(writer, sheet_name='Dataset Info', index=False)

    # Sheet 2: Segmentation Results
    seg_df = pd.DataFrame([
        {'Model': 'U-Net', 'Dice': unet_res['test_dice'], 'IoU': unet_res['test_iou']},
        {'Model': 'DeepLabV3+', 'Dice': deeplab_res['test_dice'], 'IoU': deeplab_res['test_iou']},
        *[{'Model': name, 'Dice': m['dice'], 'IoU': m['iou']} for name, m in hybrid_results.items()]
    ])
    seg_df.to_excel(writer, sheet_name='Segmentation Results', index=False)

    # Sheet 3: Classification Results
    cls_summary = pd.DataFrame({
        'Metric': ['Test Accuracy (%)', 'Best Val Accuracy (%)', 'Test Loss', 'Num Classes'],
        'Value': [cls_results['test_acc'], cls_results['best_val_acc'], cls_results['test_loss'], cls_config['num_classes']]
    })
    cls_summary.to_excel(writer, sheet_name='Classification Results', index=False)

    # Sheet 4: Version Info (NEW!)
    version_info = pd.DataFrame([{
        'Report Version': get_version_timestamp(),
        'Generated Date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'Best Segmentation Model': best_seg_model,
        'Best Dice': best_dice,
        'Classification Accuracy': cls_results['test_acc']
    }])
    version_info.to_excel(writer, sheet_name='Version Info', index=False)

print(f"✅ Saved: {excel_filename}")

In [ ]:
# ==============================================================================
# Ô CODE 7: CREATE TEXT REPORT WITH VERSIONING
# ==============================================================================
print("\n" + "="*80)
print("📝 TẠO BÁO CÁO VĂN BẢN (WITH VERSIONING)")
print("="*80)

report_version = get_version_timestamp()
report_content = f"""
{'='*80}
ĐỒ ÁN: HỆ THỐNG CHẨN ĐOÁN BỆNH DA LIỄU
{'='*80}

📅 Report Version: {report_version}
🕐 Generated: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}

{'='*80}
1. DATASET INFORMATION
{'='*80}

Segmentation Dataset:
  • Total: {seg_info['total']:,} samples
  • Train/Val/Test: {seg_info['train']}/{seg_info['val']}/{seg_info['test']}

Classification Dataset:
  • Total: {cls_info['total']:,} samples
  • Train/Val/Test: {cls_info['train']}/{cls_info['val']}/{cls_info['test']}
  • Classes: {len(cls_info['classes'])} ({', '.join(cls_info['classes'])})

{'='*80}
2. SEGMENTATION RESULTS
{'='*80}

U-Net (ResNet34):
  • Test Dice: {unet_res['test_dice']:.4f}
  • Test IoU: {unet_res['test_iou']:.4f}

DeepLabV3+ (ResNet50):
  • Test Dice: {deeplab_res['test_dice']:.4f}
  • Test IoU: {deeplab_res['test_iou']:.4f}

Hybrid Models:
"""

for model_name, metrics in hybrid_results.items():
    report_content += f"  {model_name}:\n    - Dice: {metrics['dice']:.4f}\n    - IoU: {metrics['iou']:.4f}\n"

dice_improvement = ((best_dice - unet_res['test_dice']) / unet_res['test_dice'] * 100)

report_content += f"""
🏆 BEST MODEL: {best_seg_model}
  • Dice: {best_dice:.4f}
  • IoU: {best_iou:.4f}
  • Improvement: {dice_improvement:+.2f}%

{'='*80}
3. ROI EXTRACTION
{'='*80}

Source Model: {all_checkpoints["05"]["source_model"]}
Statistics:
  • Train: {roi_info['train']:,} ROIs
  • Val: {roi_info['val']:,} ROIs
  • Test: {roi_info['test']:,} ROIs
  • Total: {roi_info['train'] + roi_info['val'] + roi_info['test']:,} ROIs

{'='*80}
4. CLASSIFICATION RESULTS
{'='*80}

Model: EfficientNet-B0 + CBAM Attention
Results:
  • Test Accuracy: {cls_results['test_acc']:.2f}%
  • Best Val Accuracy: {cls_results['best_val_acc']:.2f}%
  • Test Loss: {cls_results['test_loss']:.4f}
  • Epochs Trained: {cls_config['epochs_trained']}

{'='*80}
5. VERSION HISTORY
{'='*80}

Current Version: {report_version}
Archived Reports: {RESULTS_ARCHIVE_PATH}

To view previous versions:
  ls {RESULTS_ARCHIVE_PATH}

{'='*80}
END OF REPORT
{'='*80}
"""

report_filename = "FINAL_REPORT.txt"
save_with_versioning(report_content, report_filename, archive=True)

print(f"📄 Report created with version: {report_version}")

In [ ]:
# ==============================================================================
# Ô CODE 8: LIST ARCHIVED VERSIONS
# ==============================================================================
print("\n" + "="*80)
print("📚 LỊCH SỬ CÁC PHIÊN BẢN BÁO CÁO")
print("="*80)

if os.path.exists(RESULTS_ARCHIVE_PATH):
    archived_files = sorted(os.listdir(RESULTS_ARCHIVE_PATH))

    if archived_files:
        print(f"Found {len(archived_files)} archived versions:\n")
        for i, filename in enumerate(archived_files, 1):
            filepath = os.path.join(RESULTS_ARCHIVE_PATH, filename)
            size = os.path.getsize(filepath) / 1024  # KB
            print(f"  {i:2d}. {filename} ({size:.1f} KB)")
    else:
        print("📌 No archived versions yet. This is the first report!")
else:
    print("📁 Archive folder will be created on first run.")

print(f"\n📂 Archive location: {RESULTS_ARCHIVE_PATH}")

In [ ]:
# ==============================================================================
# Ô CODE 9: SAVE FINAL CHECKPOINT
# ==============================================================================
print("\n" + "="*80)
print("💾 LƯU FINAL CHECKPOINT")
print("="*80)

final_checkpoint = {
    "step": "07_final_evaluation",
    "status": "completed",
    "version": report_version,
    "timestamp": datetime.now().isoformat(),
    "summary": {
        "best_segmentation": {
            "model": best_seg_model,
            "dice": float(best_dice),
            "iou": float(best_iou)
        },
        "classification": {
            "test_accuracy": float(cls_results['test_acc']),
            "best_val_accuracy": float(cls_results['best_val_acc'])
        }
    },
    "output_files": {
        "text_report": os.path.join(RESULTS_PATH, "FINAL_REPORT.txt"),
        "excel_summary": excel_path,
        "segmentation_comparison": seg_comparison_path,
        "pipeline_performance": pipeline_summary_path
    },
    "archive_path": RESULTS_ARCHIVE_PATH
}

checkpoint_path = os.path.join(CHECKPOINT_PATH, "07_final_evaluation_complete.json")
with open(checkpoint_path, 'w') as f:
    json.dump(final_checkpoint, f, indent=4)

print(f"✅ Checkpoint saved: {checkpoint_path}")

In [ ]:
# ==============================================================================
# Ô CODE 9.5: FIX BUG #7 - ADD PER-CLASS METRICS (Classification Evaluation)
# ==============================================================================

print("\n" + "="*80)
print("🧪 TEST EVALUATION WITH PER-CLASS METRICS (FIX BUG #7)")
print("="*80)

# ==============================================================================
# LOAD TEST DATA
# ==============================================================================

print("\n📦 LOADING TEST DATA...")

ROI_OUTPUT_PATH = os.path.join(GDRIVE_PATH, "1_Data/processed/roi_data")
test_csv_path = os.path.join(ROI_OUTPUT_PATH, "test", "test.csv")

test_df = pd.read_csv(test_csv_path)

# Get classes
classes = cls_info['classes']
num_classes = len(classes)
class_to_idx = {cls: i for i, cls in enumerate(classes)}
idx_to_class = {i: cls for i, cls in enumerate(classes)}

print(f"✅ Test data loaded: {len(test_df)} samples")
print(f"   Classes: {classes}")

# ==============================================================================
# DEFINE ROI DATASET & DATALOADER
# ==============================================================================

class ROIDataset(Dataset):
    def __init__(self, df, class_to_idx, transform=None):
        self.df = df.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row['roi_path'])
        if img is None:
            raise ValueError(f"Cannot load: {row['roi_path']}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)

        if self.transform:
            img = self.transform(img)

        label = self.class_to_idx[row['dx']]
        return img, label

# Transform
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

test_dataset = ROIDataset(test_df, class_to_idx, transform=val_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"✅ Test DataLoader created: {len(test_loader)} batches")

# ==============================================================================
# LOAD BEST MODEL
# ==============================================================================

print("\n📦 LOADING CLASSIFICATION MODEL...")

from PIL import Image

# Define model classes (copy từ File 06)
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg, max_out], dim=1)
        return self.sigmoid(self.conv(x))

class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.channel = ChannelAttention(in_channels, reduction)
        self.spatial = SpatialAttention()

    def forward(self, x):
        x = x * self.channel(x)
        return x * self.spatial(x)

class EfficientNetWithAttention(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=pretrained, num_classes=0)
        self.attention = CBAM(self.backbone.num_features)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self.backbone.num_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone.forward_features(x)
        features = self.attention(features)
        features = self.global_pool(features).flatten(1)
        return self.classifier(features)

model = EfficientNetWithAttention(num_classes=num_classes, pretrained=True)

BEST_MODEL_PATH = os.path.join(GDRIVE_PATH, "4_Models/classification/efficientnet_attention_best.pth")
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print(f"✅ Model loaded from: {BEST_MODEL_PATH}")

# ==============================================================================
# GENERATE PREDICTIONS (FIX BUG #7)
# ==============================================================================

print("\n📊 GENERATING TEST PREDICTIONS...")

test_preds_list = []
test_labels_list = []
test_probs_list = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Test inference"):
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)

        test_probs_list.append(probs.cpu().numpy())
        test_preds_list.append(preds.cpu().numpy())
        test_labels_list.append(labels.numpy())

test_preds = np.concatenate(test_preds_list)
test_labels = np.concatenate(test_labels_list)
test_probs = np.concatenate(test_probs_list)

overall_accuracy = accuracy_score(test_labels, test_preds)

print(f"✅ Predictions completed:")
print(f"   Test samples: {len(test_labels)}")
print(f"   Overall Accuracy: {overall_accuracy*100:.2f}%")

# ==============================================================================
# CALCULATE PER-CLASS METRICS (FIX BUG #7)
# ==============================================================================

print("\n" + "="*80)
print("📋 PER-CLASS METRICS (FIX BUG #7)")
print("="*80)

precision, recall, f1, support = precision_recall_fscore_support(
    test_labels, test_preds, labels=range(num_classes), zero_division=0
)

# Classification report
print("\n" + classification_report(
    test_labels, test_preds,
    target_names=classes,
    digits=4,
    zero_division=0
))

# ==============================================================================
# CALCULATE SPECIFICITY (FIX BUG #7)
# ==============================================================================

print("\n" + "="*80)
print("📊 SENSITIVITY & SPECIFICITY (FIX BUG #7)")
print("="*80)

specificity_list = []

for class_idx in range(num_classes):
    sensitivity = recall[class_idx]
    tn = np.sum((test_labels != class_idx) & (test_preds != class_idx))
    fp = np.sum((test_labels != class_idx) & (test_preds == class_idx))
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    specificity_list.append(specificity)

sensitivity_spec_df = pd.DataFrame({
    'Class': classes,
    'Sensitivity (Recall)': [f"{r:.4f}" for r in recall],
    'Specificity': [f"{s:.4f}" for s in specificity_list],
    'Balanced Accuracy': [(recall[i] + specificity_list[i]) / 2 for i in range(num_classes)]
})

print("\n" + sensitivity_spec_df.to_string(index=False))

# ==============================================================================
# CALCULATE AUC-ROC PER CLASS (FIX BUG #7)
# ==============================================================================

print("\n" + "="*80)
print("📊 AUC-ROC PER CLASS (FIX BUG #7)")
print("="*80)

test_labels_onehot = label_binarize(test_labels, classes=range(num_classes))

auc_roc_list = []
fpr_dict = {}
tpr_dict = {}

for class_idx in range(num_classes):
    try:
        auc = roc_auc_score(test_labels_onehot[:, class_idx], test_probs[:, class_idx])
        auc_roc_list.append(auc)

        fpr, tpr, _ = roc_curve(test_labels_onehot[:, class_idx], test_probs[:, class_idx])
        fpr_dict[class_idx] = fpr
        tpr_dict[class_idx] = tpr

        print(f"✅ {classes[class_idx]:10s}: AUC-ROC = {auc:.4f}")
    except Exception as e:
        print(f"⚠️ {classes[class_idx]:10s}: Cannot calculate AUC-ROC")
        auc_roc_list.append(0.0)

macro_auc = np.mean([a for a in auc_roc_list if a > 0])
print(f"\n📊 Macro-average AUC-ROC: {macro_auc:.4f}")

# ==============================================================================
# CONFUSION MATRIX (FIX BUG #7)
# ==============================================================================

print("\n" + "="*80)
print("🔲 CONFUSION MATRIX (FIX BUG #7)")
print("="*80)

cm = confusion_matrix(test_labels, test_preds, labels=range(num_classes))

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes, linewidths=0.5, cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Test Set (FIX BUG #7)', fontsize=16, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

confusion_matrix_path = os.path.join(RESULTS_PATH, 'confusion_matrix_bug7_fix.png')
archive_old_report(confusion_matrix_path, RESULTS_ARCHIVE_PATH)
plt.savefig(confusion_matrix_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"✅ Saved: confusion_matrix_bug7_fix.png")

# ==============================================================================
# SECTION: PREDICTION CONFIDENCE ANALYSIS (FIX BUG #18)
# ==============================================================================

print("\n" + "="*80)
print("📊 PREDICTION CONFIDENCE ANALYSIS (FIX BUG #18)")
print("="*80)

# ✅ FIX BUG #18: TÍNH CONFIDENCE (CỘ TỰ TIN) CỦA MỖI PREDICTION

print("\n📈 BƯỚC 1: TẬP HỢP CONFIDENCE SCORES")
print("-" * 80)

# Lấy max probability của mỗi prediction (= model confidence)
max_probs = test_probs.max(axis=1)  # Shape: (n_samples,)

print(f"   Tổng số predictions: {len(max_probs)}")
print(f"   Confidence range: [{max_probs.min():.4f}, {max_probs.max():.4f}]")
print(f"   Mean confidence: {max_probs.mean():.4f}")
print(f"   Median confidence: {np.median(max_probs):.4f}")
print(f"   Std Dev: {max_probs.std():.4f}")

# ✅ FIX BUG #18: VISUALIZE CONFIDENCE DISTRIBUTION
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(max_probs, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(max_probs.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {max_probs.mean():.3f}')
axes[0].axvline(np.median(max_probs), color='green', linestyle='--', linewidth=2, label=f'Median: {np.median(max_probs):.3f}')
axes[0].set_xlabel('Confidence Score', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Predictions', fontsize=12, fontweight='bold')
axes[0].set_title('Distribution of Prediction Confidence', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Cumulative
sorted_probs = np.sort(max_probs)
cumulative = np.arange(1, len(sorted_probs) + 1) / len(sorted_probs)
axes[1].plot(sorted_probs, cumulative, linewidth=2, color='steelblue')
axes[1].axvline(0.85, color='red', linestyle='--', linewidth=2, label='Clinical threshold (0.85)')
axes[1].axvline(0.70, color='orange', linestyle='--', linewidth=2, label='Review threshold (0.70)')
axes[1].set_xlabel('Confidence Score', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Cumulative Probability', fontsize=12, fontweight='bold')
axes[1].set_title('Cumulative Distribution of Confidence', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
confidence_dist_path = os.path.join(RESULTS_PATH, 'confidence_distribution_bug18.png')
plt.savefig(confidence_dist_path, dpi=150, bbox_inches='tight')
plt.close()

print(f"\n✅ Saved: confidence_distribution_bug18.png")

# ✅ FIX BUG #18: ANALYZE CONFIDENCE BY CLASS
print(f"\n📋 BƯỚC 2: CONFIDENCE THEO CLASS")
print("-" * 80)

confidence_by_class = {}

print(f"{'Class':<15} {'N Samples':<15} {'Mean Conf':<15} {'Min':<10} {'Max':<10} {'Std':<10}")
print("-" * 75)

for class_idx, cls_name in enumerate(classes):
    # Lấy samples của class này
    class_mask = test_labels == class_idx

    if class_mask.sum() > 0:
        # Lấy confidence của model khi predict class này
        # (có thể true positive hoặc false negative)
        class_probs = test_probs[class_mask, class_idx]

        mean_conf = class_probs.mean()
        min_conf = class_probs.min()
        max_conf = class_probs.max()
        std_conf = class_probs.std()

        confidence_by_class[cls_name] = {
            'mean': float(mean_conf),
            'min': float(min_conf),
            'max': float(max_conf),
            'std': float(std_conf),
            'n_samples': int(class_mask.sum())
        }

        print(f"{cls_name:<15} {int(class_mask.sum()):<15} {mean_conf:<15.4f} "
              f"{min_conf:<10.4f} {max_conf:<10.4f} {std_conf:<10.4f}")

# ✅ FIX BUG #18: ANALYZE FALSE NEGATIVES CONFIDENCE (CRITICAL!)
print(f"\n🔴 BƯỚC 3: PHÂN TÍCH FALSE NEGATIVES - CRITICAL (FIX BUG #18)")
print("-" * 80)

print(f"""
Tại sao phân tích FN confidence?
──────────────────────────────────────────────────────────────────────────────
• False Negative (FN) = Model MISS predicted cancer as benign
• High confidence FN = Model CONFIDENTLY (95%+) said benign when actually CANCER
  → NGUY HIỂM! Model sáng trỏng → bác sĩ tin tưởng → bệnh nhân không được check
• Low confidence FN = Model unsure (55%) → bác sĩ biết cần check kỹ hơn
  → Ít nguy hiểm hơn (bác sĩ tỏ cảnh báo)

Y khoa decision:
• High conf FN → Model SAI NGUY HIỂM → Cần retrain
• Low conf FN → Model UNSURE → Chấp nhận được
""")

fn_analysis_by_class = {}

for class_idx, cls_name in enumerate(classes):
    if cls_name in malignant_classes:  # Chỉ phân tích ung thư
        # Lấy samples của class này trong ground truth
        class_mask = test_labels == class_idx

        if class_mask.sum() > 0:
            # False negatives = ground truth là class này, nhưng predict khác
            fn_mask = class_mask & (test_preds != class_idx)

            if fn_mask.sum() > 0:
                # Confidence của model cho class này (khi nó SÁNG)
                fn_probs = test_probs[fn_mask, class_idx]

                mean_fn_conf = fn_probs.mean()
                max_fn_conf = fn_probs.max()
                min_fn_conf = fn_probs.min()

                fn_count = fn_mask.sum()
                total_count = class_mask.sum()
                fn_rate = (fn_count / total_count) * 100

                fn_analysis_by_class[cls_name] = {
                    'total_samples': int(total_count),
                    'fn_count': int(fn_count),
                    'fn_rate': float(fn_rate),
                    'mean_fn_confidence': float(mean_fn_conf),
                    'max_fn_confidence': float(max_fn_conf),
                    'min_fn_confidence': float(min_fn_conf)
                }

                print(f"\n🔴 {cls_name} (UNG THƯ):")
                print(f"   Tổng samples: {total_count}")
                print(f"   False Negatives (MISS): {fn_count} ({fn_rate:.1f}%)")
                print(f"   Mean confidence of FN: {mean_fn_conf:.4f}")
                print(f"   Max confidence of FN: {max_fn_conf:.4f}")
                print(f"   Min confidence of FN: {min_fn_conf:.4f}")

                # CẢNH BÁO
                if fn_rate > 10:
                    print(f"\n   🔴 CẢNH BÁO: {fn_rate:.1f}% bệnh nhân MISS!")
                    print(f"      Ý nghĩa: {int(fn_count)} bệnh nhân {cls_name} không được phát hiện")
                    print(f"      Hậu quả: Những người này không nhận điều trị")

                if max_fn_conf > 0.80:
                    print(f"\n   🔴 NGUY HIỂM: High confidence false negative!")
                    print(f"      Model nói benign (conf: {max_fn_conf:.3f}) nhưng thực tế CANCER")
                    print(f"      Hậu quả: Bác sĩ tin model → không check kỹ → miss bệnh")
                    print(f"      Y khoa: KHÔNG AN TOÀN")
                else:
                    print(f"\n   ✅ Tốt: FN có confidence thấp ({max_fn_conf:.3f})")
                    print(f"      Bác sĩ sẽ cảnh báo (model unsure) → check kỹ hơn")
            else:
                print(f"\n✅ {cls_name}: NO FALSE NEGATIVES (PERFECT!)")
                fn_analysis_by_class[cls_name] = {
                    'total_samples': int(class_mask.sum()),
                    'fn_count': 0,
                    'fn_rate': 0.0,
                    'mean_fn_confidence': 0.0,
                    'max_fn_confidence': 0.0,
                    'min_fn_confidence': 0.0
                }

# ✅ FIX BUG #18: SET CLINICAL CONFIDENCE THRESHOLDS
print(f"\n{'='*80}")
print(f"📋 BƯỚC 4: CLINICAL CONFIDENCE THRESHOLDS (FIX BUG #18)")
print(f"{'='*80}")

clinical_thresholds = """
THRESHOLDS KHUYÊN CÁO CHO BÁC SĨ:
──────────────────────────────────────────────────────────────────────────────

🔴 MALIGNANT CLASSES (MEL, BCC - Ung thư):
──────────────────────────────────────────────────────────────────────────────
✅ Confidence ≥ 0.85:
   → BÁC SĨ: "Model rất tự tin là ung thư"
   → HÀNH ĐỘNG: Sắp lịch sinh thiết NGAY TRONG TUẦN
   → PHẢ VERIFY: Lâm sàng (dermoscopy), nhưng sinh thiết gần chắc chắn

⚠️ 0.70 ≤ Confidence < 0.85:
   → BÁC SĨ: "Model có khả năng là ung thư, nhưng không hoàn toàn chắc"
   → HÀNH ĐỘNG: Cần xem lâm sàng kỹ, dùng dermoscopy
   → PHẢ DECIDE: Sinh thiết vs. follow-up tùy vào clinical findings

❌ Confidence < 0.70:
   → BÁC SĨ: "Model không chắc, khả năng lớn là lành"
   → HÀNH ĐỘNG: Theo dõi thay đổi, không sinh thiết ngay (trừ lâm sàng nghi ngờ)
   → NHẬN XÉT: AI không đủ confident, cần bác sĩ quyết định

PRE-MALIGNANT CLASSES (AKIEC - Tiền-ung thư):
──────────────────────────────────────────────────────────────────────────────
✅ Confidence ≥ 0.80:
   → BÁC SĨ: "Tiền-ung thư - cần treatment"
   → HÀNH ĐỘNG: Tư vấn treatment options (cryotherapy, topical v.v.)

⚠️ 0.60 ≤ Confidence < 0.80:
   → BÁC SĨ: "Có khả năng tiền-ung thư"
   → HÀNH ĐỘNG: Cần observation hoặc biopsy để confirm

BENIGN CLASSES (NV, BKL, v.v. - Lành tính):
──────────────────────────────────────────────────────────────────────────────
✅ Confidence ≥ 0.80:
   → BÁC SĨ: "Model chắc là lành"
   → HÀNH ĐỘNG: Reassure bệnh nhân, theo dõi thay đổi theo protocol

⚠️ 0.60 ≤ Confidence < 0.80:
   → BÁC SĨ: "Có khả năng lành, nhưng cần kiểm tra lâm sàng"
   → HÀNH ĐỘNG: Dermoscopy, theo dõi ≥ 1 tháng

❌ Confidence < 0.60:
   → BÁC SĨ: "Model không chắc - KHÔNG THỂ TIN"
   → HÀNH ĐỘNG: Tính toán lâm sàng, có thể sinh thiết để confirm
"""

print(clinical_thresholds)

# ✅ FIX BUG #18: CATEGORIZE PREDICTIONS BY CONFIDENCE
print(f"\n{'='*80}")
print(f"📊 BƯỚC 5: CATEGORICAL ANALYSIS (FIX BUG #18)")
print(f"{'='*80}")

confidence_categories = {
    'Very High (≥0.90)': (max_probs >= 0.90).sum(),
    'High (0.80-0.90)': ((max_probs >= 0.80) & (max_probs < 0.90)).sum(),
    'Medium (0.70-0.80)': ((max_probs >= 0.70) & (max_probs < 0.80)).sum(),
    'Low (0.60-0.70)': ((max_probs >= 0.60) & (max_probs < 0.70)).sum(),
    'Very Low (<0.60)': (max_probs < 0.60).sum()
}

print(f"\n{'Confidence Level':<25} {'Count':<15} {'Percentage':<15}")
print("-" * 55)

total_preds = len(max_probs)
for level, count in confidence_categories.items():
    pct = (count / total_preds) * 100
    print(f"{level:<25} {count:<15} {pct:>6.1f}%")

# ✅ SAVE ANALYSIS TO CHECKPOINT
final_checkpoint['confidence_analysis_bug18'] = {
    'overall_stats': {
        'mean_confidence': float(max_probs.mean()),
        'median_confidence': float(np.median(max_probs)),
        'std_confidence': float(max_probs.std()),
        'min_confidence': float(max_probs.min()),
        'max_confidence': float(max_probs.max())
    },
    'by_class': confidence_by_class,
    'false_negatives_analysis': fn_analysis_by_class,
    'categorical_distribution': {k: int(v) for k, v in confidence_categories.items()},
    'clinical_thresholds': {
        'malignant_high': 0.85,
        'malignant_review': 0.70,
        'benign_safe': 0.80,
        'benign_followup': 0.60
    },
    'notes': 'Confidence scores critical for clinical decision-making'
}

print(f"\n✅ Confidence analysis saved to checkpoint")

# ==============================================================================
# ROC CURVES (GIỮ NGUYÊN, SAU KHI FIX BUG #18)
# ==============================================================================

print(f"\n" + "="*80)
print(f"📈 ROC CURVES (FIX BUG #7)")
print(f"="*80)

plt.figure(figsize=(12, 8))

for class_idx in range(num_classes):
    if class_idx in fpr_dict:
        plt.plot(fpr_dict[class_idx], tpr_dict[class_idx],
                label=f'{classes[class_idx]} (AUC={auc_roc_list[class_idx]:.3f})',
                linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random (AUC=0.5)')
plt.xlabel('False Positive Rate', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=12, fontweight='bold')
plt.title('ROC Curves - Multi-class Classification', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()

roc_curves_path = os.path.join(RESULTS_PATH, 'roc_curves_final.png')
plt.savefig(roc_curves_path, dpi=150, bbox_inches='tight')
plt.close()

print(f"✅ Saved: {roc_curves_path}")

# ==============================================================================
# COMPREHENSIVE METRICS TABLE (FIX BUG #7)
# ==============================================================================

print("\n" + "="*80)
print("📋 COMPREHENSIVE METRICS TABLE (FIX BUG #7)")
print("="*80)

comprehensive_metrics = pd.DataFrame({
    'Class': classes,
    'Precision': [f"{p:.4f}" for p in precision],
    'Sensitivity': [f"{r:.4f}" for r in recall],
    'Specificity': [f"{s:.4f}" for s in specificity_list],
    'F1-Score': [f"{f:.4f}" for f in f1],
    'AUC-ROC': [f"{a:.4f}" for a in auc_roc_list],
    'Support': support.astype(int)
})

print("\n" + comprehensive_metrics.to_string(index=False))

# Save to CSV
metrics_csv_path = os.path.join(RESULTS_PATH, "per_class_metrics_bug7_fix.csv")
comprehensive_metrics.to_csv(metrics_csv_path, index=False)
print(f"\n✅ Saved: {metrics_csv_path}")

# ==============================================================================
# UPDATE FINAL CHECKPOINT WITH PER-CLASS METRICS (FIX BUG #7)
# ==============================================================================

print("\n" + "="*80)
print("💾 UPDATING CHECKPOINT WITH PER-CLASS METRICS")
print("="*80)

final_checkpoint['per_class_metrics'] = {
    classes[i]: {
        'precision': float(precision[i]),
        'recall': float(recall[i]),
        'specificity': float(specificity_list[i]),
        'f1_score': float(f1[i]),
        'auc_roc': float(auc_roc_list[i]),
        'support': int(support[i])
    }
    for i in range(num_classes)
}

final_checkpoint['overall_metrics'] = {
    'accuracy': float(overall_accuracy),
    'macro_auc_roc': float(macro_auc)
}

final_checkpoint['output_files']['confusion_matrix'] = confusion_matrix_path
final_checkpoint['output_files']['roc_curves'] = roc_curves_path
final_checkpoint['output_files']['per_class_metrics_csv'] = metrics_csv_path

checkpoint_path = os.path.join(CHECKPOINT_PATH, "07_final_evaluation_complete.json")
with open(checkpoint_path, 'w') as f:
    json.dump(final_checkpoint, f, indent=4)

print(f"✅ Updated checkpoint: {checkpoint_path}")

print("\n" + "="*80)
print("✅ BUG #7 FIX COMPLETE - Per-class metrics added!")
print("="*80)

In [ ]:
# ==============================================================================
# Ô CODE 9.6: FIX BUG #9 - MEDICAL INTERPRETATION FOR CLASSIFICATION
# ==============================================================================

print("\n" + "="*80)
print("🏥 MEDICAL INTERPRETATION & RECOMMENDATIONS (FIX BUG #9)")
print("="*80)

# ==============================================================================
# SECTION 1: Malignant Class Sensitivity Analysis (FIX BUG #17)
# ==============================================================================

print("\n⚠️ MALIGNANT CLASS SENSITIVITY ANALYSIS (FIX BUG #17):")
print("="*80)

malignant_classes = ['MEL', 'BCC']
pre_malignant_classes = ['AKIEC']
benign_classes = ['NV', 'BKL', 'DF', 'VASC']

# ✅ FIX BUG #17: LOAD IMBALANCE SEVERITY TỪ FILE 01
print("\n📊 LOADING IMBALANCE INFO FROM FILE 01 (FIX BUG #17):")

# Lấy imbalance info từ checkpoint_step1 (đã load ở CODE 3)
imbalance_analysis = checkpoint_step1.get('imbalance_analysis', {})
imbalance_severity = imbalance_analysis.get('severity', 'UNKNOWN')
imbalance_ratio = imbalance_analysis.get('imbalance_ratio', 1.0)

print(f"   Imbalance Severity: {imbalance_severity}")
print(f"   Imbalance Ratio (Max/Min): {imbalance_ratio:.2f}x")

# ✅ FIX BUG #17: ĐỘNG LỰA CHỌN THRESHOLD DỰA VÀO SEVERITY
# Lý do Y khoa:
#   - Imbalance cao → model khó học minority class → cần threshold cao để an toàn
#   - Imbalance thấp → model dễ học → threshold thấp hơn được
#   - Ưu tiên: sensitivity (không miss cancer) > specificity

sensitivity_thresholds = {
    'EXTREME': 0.90,      # Imbalance > 15x: PHẢI 90% sensitivity (rất khó miss cancer)
    'SEVERE': 0.85,       # Imbalance 5-15x: PHẢI 85% sensitivity
    'MODERATE': 0.80,     # Imbalance 2-5x: 80% sensitivity
    'MILD': 0.75,         # Imbalance < 2x: 75% có thể đủ
    'UNKNOWN': 0.80       # Default: 80%
}

sensitivity_threshold = sensitivity_thresholds.get(imbalance_severity, 0.80)
specificity_threshold = 0.85  # Luôn cần specificity tốt để tránh false alarm

print(f"\n✅ THRESHOLDS (FIX BUG #17):")
print(f"{'='*80}")
print(f"Imbalance Severity: {imbalance_severity}")
print(f"Imbalance Ratio: {imbalance_ratio:.2f}x")
print(f"\n📋 Sensitivity Threshold: {sensitivity_threshold*100:.0f}%")
print(f"   Specificity Threshold: {specificity_threshold*100:.0f}%")
print(f"\n💡 Giải thích:")

explanation_map = {
    'EXTREME': """
   - Imbalance rất cao (>15x) → MEL có thể chỉ 0.5% của dữ liệu
   - Model khó học melanoma → phải yêu cầu sensitivity 90%
   - Nếu sensitivity < 90%: có 10% bệnh nhân sẽ bị miss (không được phát hiện)
   - Y khoa: CẦN PHẢI CẢN THẬN khi dữ liệu rất imbalance
   """,
    'SEVERE': """
   - Imbalance cao (5-15x) → MEL có thể 1-2% của dữ liệu
   - Model khó học → cần sensitivity 85%
   - Nếu sensitivity < 85%: 15% bệnh nhân miss → rủi ro cao
   """,
    'MODERATE': """
   - Imbalance trung bình (2-5x) → MEL có 5-20% của dữ liệu
   - Model học được → sensitivity 80% là chấp nhận được
   - Vẫn có 20% miss rate → cần giám sát kỹ
   """,
    'MILD': """
   - Imbalance thấp (<2x) → dữ liệu cân bằng tương đối
   - Model dễ học → sensitivity 75% là chấp nhận được
   """,
    'UNKNOWN': """
   - Không biết imbalance → dùng default 80% (an toàn)
   - Khuyên: Chạy lại File 01 để biết severity chính xác
   """
}

print(explanation_map.get(imbalance_severity, ""))

# ✅ FIX BUG #17: PHÂN TÍCH TỪNG MALIGNANT CLASS
print(f"\n{'='*80}")
print(f"PHÂN TÍCH CÁC CLASS UNG THƯ (FIX BUG #17):")
print(f"{'='*80}")

malignant_performance = []

for class_idx, cls_name in enumerate(classes):
    if cls_name in malignant_classes:
        sensitivity = recall[class_idx]
        specificity = specificity_list[class_idx]
        f1_score = f1[class_idx]

        # ✅ Kiểm tra xem có đạt threshold không
        is_acceptable = sensitivity >= sensitivity_threshold
        miss_rate = (1 - sensitivity) * 100
        fp_rate = (1 - specificity) * 100

        print(f"\n🔴 {cls_name} - UNG THƯ DA (MALIGNANT - RẤT NGUY HIỂM):")
        print(f"{'='*80}")
        print(f"   Sensitivity (Tỷ lệ phát hiện đúng): {sensitivity:.4f} ({sensitivity*100:.1f}%)")
        print(f"   Specificity (Tỷ lệ âm tính đúng):  {specificity:.4f} ({specificity*100:.1f}%)")
        print(f"   Precision (Độ chính xác):          {precision[class_idx]:.4f}")
        print(f"   F1-Score (Cân bằng):              {f1_score:.4f}")
        print(f"   AUC-ROC:                          {auc_roc_list[class_idx]:.4f}")

        print(f"\n   📊 Status: {'✅ ĐẠT NGƯỠNG' if is_acceptable else '🔴 KHÔNG ĐẠT NGƯỠNG'}")

        # ✅ PHÂN TÍCH CHI TIẾT
        if sensitivity < sensitivity_threshold:
            print(f"\n   🔴 CẢNH BÁO SENSITIVITY (CÓ NGUY HIỂM):")
            print(f"      • Tỷ lệ bỏ sót: {miss_rate:.1f}%")
            print(f"      • Ý nghĩa: Trong {100} bệnh nhân {cls_name}, {int(miss_rate)} người KHÔNG được phát hiện")
            print(f"      • Hậu quả: Những người này không nhận được điều trị → bệnh tiến triển")
            print(f"      • Y khoa: ĐỨT KHOÁT - CÓ THỂ CHẾT")
            print(f"      • Nguyên nhân: Imbalance dữ liệu {imbalance_severity} → model khó học {cls_name}")
            print(f"      • Khuyên: KHÔNG NÊN dùng hệ thống này cho {cls_name} detection")

            acceptable_status = "❌ KHÔNG CHẤP NHẬN"
        else:
            diff_from_threshold = sensitivity - sensitivity_threshold
            print(f"\n   ✅ SENSITIVITY CHẤP NHẬN:")
            print(f"      • Tỷ lệ phát hiện: {sensitivity*100:.1f}%")
            print(f"      • Vượt ngưỡng: +{diff_from_threshold*100:.1f}% so với {sensitivity_threshold*100:.0f}%")
            print(f"      • Ý nghĩa: Trong 100 bệnh nhân {cls_name}, {int(sensitivity*100)} người được phát hiện")
            print(f"      • Y khoa: ĐỦ AN TOÀN để dùng làm screening tool")
            print(f"      • Khuyên: CÓ THỂ dùng hệ thống như công cụ hỗ trợ chẩn đoán")

            acceptable_status = "✅ CHẤP NHẬN"

        # Specificity analysis
        if specificity < specificity_threshold:
            print(f"\n   ⚠️ CẢNH BÁO SPECIFICITY (RỦI RO CAO):")
            print(f"      • Tỷ lệ dương tính giả: {fp_rate:.1f}%")
            print(f"      • Ý nghĩa: {int(fp_rate)} bệnh nhân lành tính sẽ được chẩn đoán nhầm là {cls_name}")
            print(f"      • Hậu quả: Sinh thiết không cần thiết (xâm lấn, chi phí)")
            print(f"      • Mức độ: TRUNG BÌNH (không nguy hiểm như miss cancer, nhưng báo động sai)")
        else:
            print(f"\n   ✅ SPECIFICITY CHẤP NHẬN ({specificity*100:.1f}%)")

        print(f"\n   📋 KẾT LUẬN: {acceptable_status}")

        malignant_performance.append({
            'class': cls_name,
            'sensitivity': sensitivity,
            'specificity': specificity,
            'f1_score': f1_score,
            'acceptable': is_acceptable,
            'miss_rate': float(miss_rate),
            'threshold_used': sensitivity_threshold,
            'imbalance_severity': imbalance_severity
        })

# ✅ TỔNG HỢP: CÓ AN TOÀN DÙNG KHÔNG?
print(f"\n{'='*80}")
print(f"💼 KẾT LUẬN: CÓ AN TOÀN DÙNG CHO LÂM SÀNG KHÔNG?")
print(f"{'='*80}")

all_acceptable = all([m['acceptable'] for m in malignant_performance])
unacceptable_classes = [m['class'] for m in malignant_performance if not m['acceptable']]

if all_acceptable:
    print(f"""
✅ CÓ AN TOÀN - CHẤP THUẬN DÙNG HỆ THỐNG

Lý do:
  • Tất cả các class ung thư đều đạt ngưỡng sensitivity yêu cầu
  • Hệ thống có thể dùng làm công cụ hỗ trợ chẩn đoán
  • Giúp bác sĩ phát hiện sớm các trường hợp nghi ngờ

Cảnh báo:
  • KHÔNG THỂ thay thế chẩn đoán của bác sĩ da liễu
  • LUÔN PHẢ CONFIRM bằng sinh thiết (histopathology)
  • Nếu nghi ngờ cao lâm sàng, SINH THIẾT ngay kể cả AI nói benign
  • Giám sát thường xuyên các false negative cases
""")
else:
    print(f"""
❌ KHÔNG AN TOÀN - KHÔNG NÊN DÙNG

Lý do:
  • Các class sau KHÔNG đạt ngưỡng sensitivity: {', '.join(unacceptable_classes)}
  • Tỷ lệ bỏ sót quá cao → rủi ro bệnh nhân không được phát hiện
  • Hệ thống không đủ độ tin cậy

Khuyến cáo:
  1. Tăng dữ liệu training cho class có vấn đề (special data collection)
  2. Thử balanced loss function mạnh hơn (Focal Loss, OHEM)
  3. Thử oversampling hoặc augmentation để cân bằng dữ liệu
  4. Retrain model và test lại
  5. Nếu vẫn thất bại: liên hệ chuyên gia ML/dermatology

Tạm thời:
  • KHÔNG dùng cho {', '.join(unacceptable_classes)}
  • Có thể dùng cho các class khác
  • Cần xác nhận thủ công từ bác sĩ trước khi report
""")

# ✅ LƯU VÀO CHECKPOINT
final_checkpoint['malignant_sensitivity_analysis'] = {
    'imbalance_severity_used': imbalance_severity,
    'dynamic_sensitivity_threshold': sensitivity_threshold,
    'specific_threshold': specificity_threshold,
    'thresholds_explanation': explanation_map.get(imbalance_severity, ""),
    'all_malignant_acceptable': all_acceptable,
    'unacceptable_classes': unacceptable_classes,
    'malignant_performance_details': [
        {
            'class': m['class'],
            'sensitivity': float(m['sensitivity']),
            'specificity': float(m['specificity']),
            'f1_score': float(m['f1_score']),
            'miss_rate': m['miss_rate'],
            'acceptable': m['acceptable']
        }
        for m in malignant_performance
    ]
}

print(f"\n✅ Malignant analysis saved to checkpoint")

# ==============================================================================
# SECTION 2: Pre-malignant Class Assessment
# ==============================================================================

print("="*80)
print("🟡 PRE-MALIGNANT CLASS ASSESSMENT (FIX BUG #9):")
print("="*80)

for class_idx, cls_name in enumerate(classes):
    if cls_name in pre_malignant_classes:
        sensitivity = recall[class_idx]

        print(f"\n🟡 {cls_name} (PRE-MALIGNANT):")
        print(f"   ├─ Sensitivity: {sensitivity:.4f} ({sensitivity*100:.1f}%)")
        print(f"   ├─ Specificity: {specificity_list[class_idx]:.4f}")
        print(f"   └─ F1-Score:    {f1[class_idx]:.4f}")

        if sensitivity < 0.75:
            print(f"\n   ⚠️  SENSITIVITY CONCERN:")
            print(f"      • {(1-sensitivity)*100:.1f}% miss rate for pre-malignant lesions")
            print(f"      • May delay treatment of progressive lesions")
        else:
            print(f"\n   ✅ ACCEPTABLE SENSITIVITY")

print("\n")

# ==============================================================================
# SECTION 3: Clinical Recommendations by Class
# ==============================================================================

print("="*80)
print("📋 CLINICAL RECOMMENDATIONS BY CLASS (FIX BUG #9):")
print("="*80)

clinical_guidance = {
    'MEL': {
        'name': 'Melanoma',
        'urgency': 'IMMEDIATE',
        'recommendation': 'BIOPSY REQUIRED',
        'actions': [
            'Schedule dermatology consultation within 1 WEEK',
            'Prepare for biopsy/histopathology',
            'Consider oncology referral',
            'Look for ABCDE signs (Asymmetry, Border, Color, Diameter, Evolution)',
            'Do NOT delay - Melanoma is aggressive'
        ],
        'treatment_options': [
            'Surgical excision',
            'Mohs micrographic surgery',
            'Immunotherapy if advanced',
            'Radiation therapy if indicated'
        ]
    },
    'BCC': {
        'name': 'Basal Cell Carcinoma',
        'urgency': 'HIGH',
        'recommendation': 'BIOPSY RECOMMENDED',
        'actions': [
            'Schedule dermatology consultation within 2 WEEKS',
            'Discuss biopsy and treatment options',
            'Consider Mohs surgery if on face/hand',
            'Assess for recurrence risk factors'
        ],
        'treatment_options': [
            'Surgical excision',
            'Mohs micrographic surgery (preferred for face)',
            'Cryotherapy (for small, superficial lesions)',
            'Topical chemotherapy (5-FU, imiquimod) for superficial BCC'
        ]
    },
    'AKIEC': {
        'name': 'Actinic Keratosis',
        'urgency': 'MEDIUM',
        'recommendation': 'CONSIDER BIOPSY',
        'actions': [
            'Schedule dermatology consultation within 1 MONTH',
            'Assess for treatment need',
            'Establish monitoring schedule',
            'Strict sun protection'
        ],
        'treatment_options': [
            'Cryotherapy (freezing)',
            'Topical 5-Fluorouracil (5-FU)',
            'Topical imiquimod',
            'Chemical peel',
            'Laser therapy',
            'Photodynamic therapy'
        ]
    }
}

for cls_name in classes:
    if cls_name in clinical_guidance:
        guidance = clinical_guidance[cls_name]

        print(f"\n{'='*80}")
        print(f"🔬 {guidance['name'].upper()} ({cls_name})")
        print(f"{'='*80}")

        print(f"\n⏰ Urgency: {guidance['urgency']}")
        print(f"📌 Recommendation: {guidance['recommendation']}")

        print(f"\n✓ Recommended Actions:")
        for i, action in enumerate(guidance['actions'], 1):
            print(f"   {i}. {action}")

        print(f"\n💊 Treatment Options:")
        for i, option in enumerate(guidance['treatment_options'], 1):
            print(f"   {i}. {option}")

print("\n")

# ==============================================================================
# SECTION 4: System Performance Summary
# ==============================================================================

print("="*80)
print("📊 SYSTEM PERFORMANCE SUMMARY (FIX BUG #9):")
print("="*80)

print(f"\nOverall Test Accuracy: {overall_accuracy*100:.2f}%")
print(f"Macro-average AUC-ROC: {macro_auc:.4f}")

# Identify problematic classes
problematic_classes = []
for i, cls in enumerate(classes):
    if recall[i] < 0.70 or precision[i] < 0.70:
        problematic_classes.append((cls, recall[i], precision[i]))

if problematic_classes:
    print(f"\n⚠️ PROBLEMATIC CLASSES (performance < 0.70):")
    for cls, sens, prec in problematic_classes:
        print(f"   • {cls}: Sensitivity={sens:.4f}, Precision={prec:.4f}")
else:
    print(f"\n✅ All classes have acceptable performance (≥ 0.70)")

# ==============================================================================
# SECTION 5: Limitations and Recommendations
# ==============================================================================

print("\n" + "="*80)
print("⚠️ SYSTEM LIMITATIONS & RECOMMENDATIONS (FIX BUG #9):")
print("="*80)

limitations = """
TECHNICAL LIMITATIONS:
─────────────────────────────────────────────────────────────────────────────
1. Model Training Data:
   • Trained on ISIC dataset (may differ from real clinic samples)
   • Potential domain shift between training and clinical use
   • Recommendation: Validate on local clinic images

2. Skin Tone & Demographics:
   • Dataset distribution may not represent all skin tones equally
   • Performance may vary by patient demographics
   • Recommendation: Use with caution for underrepresented groups

3. Context Information:
   • Model considers only lesion image
   • Does NOT consider: patient history, location, size trend, symptoms
   • Recommendation: Always combine AI with clinical examination

4. Borderline Cases:
   • Some lesions are inherently difficult to classify
   • May require expert evaluation regardless of AI prediction
   • Recommendation: Low confidence predictions need dermatologist review

MEDICAL LIMITATIONS:
─────────────────────────────────────────────────────────────────────────────
1. This is a SCREENING tool, NOT a diagnostic tool
   • For screening: sensitivity is prioritized (detect most cases)
   • For diagnosis: requires histopathology (biopsy)

2. False Negatives (missed cancers) are MORE dangerous than false positives
   • Always biopsy if clinical suspicion is high, regardless of AI prediction
   • Sensitivity < 80% means 1-2 in 10 cancers could be missed

3. Sensitivity-Specificity Trade-off:
   • Increasing sensitivity may increase false positives
   • Clinical judgment needed to balance risks vs unnecessary treatment
"""

print(limitations)

# ==============================================================================
# SECTION 6: Recommendations for Clinical Integration
# ==============================================================================

print("="*80)
print("🏥 RECOMMENDATIONS FOR CLINICAL INTEGRATION (FIX BUG #9):")
print("="*80)

integration_recommendations = """
SUGGESTED CLINICAL WORKFLOW:
─────────────────────────────────────────────────────────────────────────────
1. USE AS SCREENING TOOL:
   ✓ Input: Patient with suspicious skin lesion
   ✓ AI output: Preliminary classification + confidence score
   ✓ Use: Decision support for further workup

2. CONFIDENCE-BASED DECISION MAKING:
   • High confidence (≥85%): May proceed with recommended action
   • Medium confidence (60-85%): Requires dermatologist review
   • Low confidence (<60%): Requires expert evaluation

3. SPECIAL HANDLING FOR MALIGNANT CLASSES:
   • MEL or BCC prediction (any confidence):
     → Recommend immediate dermatology consultation
     → Biopsy is gold standard for confirmation

4. ALWAYS REMEMBER:
   • Biopsy is the gold standard for histopathological diagnosis
   • Clinical examination is essential (dermoscopy, palpation, history)
   • Patient safety is the top priority
   • In case of doubt, refer to specialist

QUALITY ASSURANCE:
─────────────────────���───────────────────────────────────────────────────────
✓ Regular model validation on new clinical cases
✓ Track false negatives and false positives
✓ Continuous training on misclassified cases
✓ Performance monitoring by patient demographics
✓ Regular audits of clinical outcomes
"""

print(integration_recommendations)

# ==============================================================================
# SECTION 7: Save Medical Interpretation Report
# ==============================================================================

print("\n" + "="*80)
print("💾 SAVING MEDICAL INTERPRETATION REPORT (FIX BUG #9):")
print("="*80)

medical_report_content = f"""
{'='*80}
MEDICAL INTERPRETATION REPORT - CLASSIFICATION SYSTEM
{'='*80}

Report Version: {report_version}
Generated: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}

{'='*80}
1. EXECUTIVE SUMMARY
{'='*80}

Overall System Accuracy: {overall_accuracy*100:.2f}%
Macro-average AUC-ROC: {macro_auc:.4f}

Status: {'✅ READY FOR CLINICAL SCREENING' if overall_accuracy >= 0.85 else '⚠️ REQUIRES SPECIALIST REVIEW' if overall_accuracy >= 0.75 else '❌ NOT RECOMMENDED FOR CLINICAL USE'}

{'='*80}
2. PER-CLASS PERFORMANCE
{'='*80}

"""

for i, cls in enumerate(classes):
    medical_report_content += f"""
{cls} ({'Malignant' if cls in malignant_classes else 'Pre-malignant' if cls in pre_malignant_classes else 'Benign'}):
  • Sensitivity (Recall): {recall[i]:.4f}
  • Specificity: {specificity_list[i]:.4f}
  • Precision: {precision[i]:.4f}
  • F1-Score: {f1[i]:.4f}
  • AUC-ROC: {auc_roc_list[i]:.4f}
  • Support: {support[i]}
"""

medical_report_content += f"""

{'='*80}
3. MALIGNANT CLASS ANALYSIS
{'='*80}

"""

for perf in malignant_performance:
    status = "✅ ACCEPTABLE" if perf['acceptable'] else "⚠️ WARNING"
    medical_report_content += f"""
{perf['class']}: {status}
  • Sensitivity: {perf['sensitivity']*100:.1f}%
  • Specificity: {perf['specificity']*100:.1f}%
  • Assessment: {'Safe for clinical use' if perf['acceptable'] else 'Caution - risk of missing cases'}
"""

medical_report_content += f"""

{'='*80}
4. CLINICAL RECOMMENDATIONS
{'='*80}

{limitations}

{integration_recommendations}

{'='*80}
5. LIMITATIONS & DISCLAIMERS
{'='*80}

⚠️ THIS SYSTEM IS FOR SCREENING PURPOSES ONLY

• NOT a substitute for professional medical diagnosis
• Biopsy/histopathology is gold standard for confirmation
• Always consult dermatologist for final diagnosis
• In case of suspected melanoma, seek IMMEDIATE medical attention
• Do NOT delay treatment based on AI prediction

{'='*80}
END OF MEDICAL INTERPRETATION REPORT
{'='*80}
"""

# Save medical report
medical_report_path = os.path.join(RESULTS_PATH, "MEDICAL_INTERPRETATION_BUG9.txt")
archive_old_report(medical_report_path, RESULTS_ARCHIVE_PATH)

with open(medical_report_path, 'w', encoding='utf-8') as f:
    f.write(medical_report_content)

print(f"✅ Saved: MEDICAL_INTERPRETATION_BUG9.txt")

# ==============================================================================
# SECTION 8: Update Final Checkpoint with Medical Info (FIX BUG #9)
# ==============================================================================

print("\n" + "="*80)
print("💾 UPDATING CHECKPOINT WITH MEDICAL INTERPRETATION (FIX BUG #9)")
print("="*80)

final_checkpoint['medical_interpretation'] = {
    'overall_accuracy': float(overall_accuracy),
    'macro_auc_roc': float(macro_auc),
    'clinical_status': 'READY_FOR_SCREENING' if overall_accuracy >= 0.85 else 'REQUIRES_SPECIALIST_REVIEW' if overall_accuracy >= 0.75 else 'NOT_RECOMMENDED',
    'malignant_classes': malignant_classes,
    'malignant_performance': malignant_performance,
    'problematic_classes': [
        {
            'class': cls,
            'sensitivity': float(sens),
            'precision': float(prec)
        }
        for cls, sens, prec in problematic_classes
    ],
    'clinical_recommendations': 'See MEDICAL_INTERPRETATION_BUG9.txt',
    'limitations': [
        'Domain shift between ISIC training data and real clinic',
        'Potential bias by skin tone',
        'Context information (location, history) not considered',
        'Requires dermatologist confirmation for final diagnosis'
    ]
}

final_checkpoint['output_files']['medical_interpretation'] = medical_report_path

checkpoint_path = os.path.join(CHECKPOINT_PATH, "07_final_evaluation_complete.json")
with open(checkpoint_path, 'w') as f:
    json.dump(final_checkpoint, f, indent=4)

print(f"✅ Updated checkpoint with medical interpretation")

print("\n" + "="*80)
print("✅ BUG #9 FIX COMPLETE - Medical Interpretation Added to File 07!")
print("="*80)

print(f"""
📋 Medical Interpretation Summary:
   ├─ Malignant class analysis (MEL, BCC)
   ├─ Pre-malignant assessment (AKIEC)
   ├─ Clinical recommendations by class
   ├─ System limitations & disclaimers
   ├─ Clinical workflow suggestions
   └─ Report saved: MEDICAL_INTERPRETATION_BUG9.txt
""")

In [ ]:
# ==============================================================================
# Ô CODE 10: SUMMARY
# ==============================================================================
print("\n" + "="*80)
print("🎉 HOÀN THÀNH BÁO CÁO CUỐI CÙNG!")
print("="*80)

print(f"""
📊 Report Version: {report_version}

✅ Files Created/Updated:
   • FINAL_REPORT.txt (latest version)
   • FINAL_REPORT.xlsx (latest version)
   • segmentation_comparison.png
   • pipeline_performance.png

📦 Archived Versions: {len(archived_files) if 'archived_files' in locals() else 0}

📁 Locations:
   • Latest Reports: {RESULTS_PATH}
   • Archived Reports: {RESULTS_ARCHIVE_PATH}

🏆 Best Results:
   • Segmentation: {best_seg_model} (Dice: {best_dice:.4f})
   • Classification: {cls_results['test_acc']:.2f}%

💡 To compare with previous versions:
   1. Check archive folder: {RESULTS_ARCHIVE_PATH}
   2. Compare Dice/Accuracy metrics across versions
   3. Track improvements over time
""")

print("="*80)
print("✨ VERSIONING ENABLED - Tất cả lịch sử báo cáo đã được lưu!")
print("="*80)